# UAP Datathon — text-only transformer training (Colab T4)

**What this notebook does.** Fine-tunes text-only classifiers on the locked Colab fold set (fold set B)
and saves out-of-fold (OOF) and test probabilities for local, honest blending.

**What it never does.**
* It never reads `id` as a model input. `id` is loaded only to check that the test file is in
  `sample_submission` order. No ID prior, no SID, no ID head, no row-order feature.
* It never writes a submission CSV.
* It never uses leaderboard feedback. The only metric printed is grouped OOF macro-F1.

**How to run.**
1. Runtime → Change runtime type → **T4 GPU**.
2. Put `train.csv`, `test.csv`, `sample_submission.csv` and `fold_ids_colab.npy` in
   `MyDrive/uap_text/data/` (the fold file is `final_sources/fold_ids.npy` from
   `uap_complete_export_20260928_064140.zip`, renamed).
3. Run all cells. Everything is saved to Drive after each fold. After a disconnect, just run all
   cells again: finished folds are validated and skipped.
4. Run the last cell to build the ZIP for local blending.

**Runs (edit `RUN_NAMES` in the config cell).**
| run | model | rows | why |
|---|---|---|---|
| `xlmr_text` | FacebookAI/xlm-roberta-base | all | Same encoder as the ID-head model, no ID input. Gives a clean text-only ablation and the main new source. |
| `banglabert_bn` | csebuetnlp/banglabert | Bangla-script rows only | Bangla-script rows are the weakest text-only slice (macro-F1 0.45 vs 0.59 for Latin script). |
| `muril_text` | google/muril-base-cased | all | Optional third run: trained on transliterated Indic text, so it may help with Banglish. |

Rough T4 cost per full 5-fold run (4 epochs): xlmr_text ≈ 95 min, muril_text ≈ 100 min,
banglabert_bn ≈ 45 min.

In [ ]:
# ============ CELL 1: SETUP (Drive, GPU fail-fast, installs) ============
import os, sys, subprocess, json, time, hashlib, platform
from pathlib import Path

WORK_ROOT = Path(os.environ.get("UAP_WORK_ROOT", "/content/drive/MyDrive/uap_text"))
REQUIRE_GPU = os.environ.get("UAP_REQUIRE_GPU", "1") == "1"

if str(WORK_ROOT).startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=False)

import numpy as np, pandas as pd
import torch

if REQUIRE_GPU and not torch.cuda.is_available():
    raise SystemExit("NO GPU. Runtime > Change runtime type > T4 GPU, then run again.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CUDA = DEVICE == "cuda"

DATA_DIR = WORK_ROOT / "data"
RUN_DIR = WORK_ROOT / "runs"
LOG_PATH = WORK_ROOT / "progress.log"
for p in (DATA_DIR, RUN_DIR):
    p.mkdir(parents=True, exist_ok=True)


def log(msg):
    """Print and append a timestamped line to the Drive progress log."""
    line = time.strftime("%Y-%m-%d %H:%M:%S") + "  " + msg
    print(line, flush=True)
    with open(LOG_PATH, "a", encoding="utf-8") as fh:
        fh.write(line + "\n")


log(f"device={DEVICE} gpu={torch.cuda.get_device_name(0) if CUDA else '-'} "
    f"torch={torch.__version__} python={platform.python_version()} work_root={WORK_ROOT}")

In [ ]:
# ============ CELL 2: CONFIG (edit RUN_NAMES only) ============
SEED = 42
N_SPLITS = 5

# Expected hashes, all verified against the project handoff. Any mismatch stops the notebook.
EXPECTED = dict(
    train_sha256="a269ad841f824c6e60930cfda8cb25bdce58a6a1ee01fc1fe663d23b19e34449",
    test_sha256="cca36f5416abfb8a41b1c84e5fa83aec18ffbff32fecd277c668a3f73312a38f",
    sample_sha256="0f7d405dbd1e88ea285c7138514b7652f7441e995dcaed338bb208a86c2f347d",
    fold_file_sha256="5ab5cfe96a72b04c8bd30adc45c1eeeee477707b0c382a19d179d8635295fbbd",
    fold_int64_sha256="cc4fdd169110aeee6d3d5566095234461041b1a9290e2e916ce850c31d27a7ca",
    fold_sizes=[9543, 9546, 9571, 9581, 9576],
)

# One shared recipe, fixed BEFORE seeing any result (no per-model tuning):
#  - mean pooling + linear head, 5-sample multi-sample dropout during training
#  - layer-wise LR decay 0.9, head LR 1e-4, AdamW wd 0.01 (no decay on bias/LayerNorm)
#  - class-balanced cross-entropy (weights from fold-train counts) + label smoothing 0.05
#  - 4 epochs, linear schedule, 6% warmup, fp16 autocast, grad clip 1.0
#  - prediction = mean of the last 2 epochs' probabilities (fixed rule, no best-epoch picking)
BASE = dict(max_len=128, epochs=4, bs=32, lr=3e-5, head_lr=1e-4, llrd=0.9, wd=0.01,
            warmup=0.06, label_smoothing=0.05, msd=5, dropout=0.1, avg_last=2,
            grad_accum=1, norm="basic", subset=None, save_state=False)

RUNS = {
    "xlmr_text": dict(BASE, model="FacebookAI/xlm-roberta-base"),
    "banglabert_bn": dict(BASE, model="csebuetnlp/banglabert", subset="bangla", norm="csebuet"),
    "muril_text": dict(BASE, model="google/muril-base-cased"),
}

RUN_NAMES = ["xlmr_text", "banglabert_bn"]   # add "muril_text" for the optional third run

In [ ]:
# ============ CELL 3: DATA + HARD VERIFICATION ============
import re, unicodedata


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


paths = dict(train=DATA_DIR / "train.csv", test=DATA_DIR / "test.csv",
             sample=DATA_DIR / "sample_submission.csv", folds=DATA_DIR / "fold_ids_colab.npy")
for k, p in paths.items():
    if not p.exists():
        raise SystemExit(f"missing {p} - copy it into {DATA_DIR}")

DATA_HASHES = dict(train_sha256=sha256_file(paths["train"]), test_sha256=sha256_file(paths["test"]),
                   sample_sha256=sha256_file(paths["sample"]), fold_file_sha256=sha256_file(paths["folds"]))
for k, v in DATA_HASHES.items():
    assert v == EXPECTED[k], f"HASH MISMATCH {k}: {v} != {EXPECTED[k]}"

train = pd.read_csv(paths["train"], encoding="utf-8")
test = pd.read_csv(paths["test"], encoding="utf-8")
sample = pd.read_csv(paths["sample"])
assert list(train.columns) == ["id", "text", "y"] and list(test.columns) == ["id", "text"]
assert np.array_equal(test["id"].to_numpy(), sample["id"].to_numpy()), "test order != sample order"
for df in (train, test):
    df["text"] = df["text"].astype("string").fillna("").astype(str)
y = train["y"].to_numpy(dtype=np.int64)
assert set(np.unique(y)) == {0, 1, 2}
N_TR, N_TE = len(train), len(test)

folds = np.load(paths["folds"]).astype(np.int64)
assert folds.shape == (N_TR,)
assert hashlib.sha256(folds.tobytes()).hexdigest() == EXPECTED["fold_int64_sha256"]
assert np.bincount(folds, minlength=N_SPLITS).tolist() == EXPECTED["fold_sizes"]

URL_RE = re.compile(r"(?:https?://|www\.)\S+", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
WS_RE = re.compile(r"\s+", flags=re.UNICODE)
# Invisible characters to drop. ZWJ/ZWNJ (U+200C/U+200D) are kept: Bangla uses them for conjuncts.
INVIS_RE = re.compile("[​⁠﻿­]")


def clean_text(t):
    """Grouping cleaner, identical to the one used to build the folds."""
    s = "" if t is None else str(t)
    s = URL_RE.sub(" URL ", s)
    s = MENTION_RE.sub(" USER ", s)
    return WS_RE.sub(" ", s).strip()


def basic_norm(t):
    """Model input: grouping cleaner + NFKC + invisible-character removal. Emojis are preserved."""
    s = unicodedata.normalize("NFKC", clean_text(t))
    return WS_RE.sub(" ", INVIS_RE.sub("", s)).strip()


clean_tr = [clean_text(t) for t in train["text"]]
gkey = pd.Index([c.lower().strip() for c in clean_tr], dtype="object")
gcode = pd.factorize(gkey, sort=False)[0]
n_cross = int((pd.DataFrame({"g": gcode, "f": folds}).groupby("g")["f"].nunique() > 1).sum())
assert n_cross == 0, f"{n_cross} duplicate-text groups cross folds"

BN_RE = re.compile("[ঀ-৿]")
BANGLA_TR = np.array([bool(BN_RE.search(t)) for t in train["text"]])
BANGLA_TE = np.array([bool(BN_RE.search(t)) for t in test["text"]])

log(f"data OK: train {N_TR} test {N_TE} classes {np.bincount(y).tolist()} "
    f"folds {np.bincount(folds).tolist()} crossing-groups {n_cross} "
    f"bangla-script rows train {BANGLA_TR.sum()} test {BANGLA_TE.sum()}")

In [ ]:
# ============ CELL 4: MODEL + TRAINING CODE ============
import gc, math
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import transformers


def macro_f1(yt, pred):
    cm = np.bincount(np.asarray(yt) * 3 + np.asarray(pred), minlength=9).reshape(3, 3)
    tp = np.diag(cm).astype(float)
    p, r = tp / np.maximum(cm.sum(0), 1), tp / np.maximum(cm.sum(1), 1)
    return float((2 * p * r / np.maximum(p + r, 1e-12)).mean())


def per_class_f1(yt, pred):
    cm = np.bincount(np.asarray(yt) * 3 + np.asarray(pred), minlength=9).reshape(3, 3)
    tp = np.diag(cm).astype(float)
    p, r = tp / np.maximum(cm.sum(0), 1), tp / np.maximum(cm.sum(1), 1)
    return [round(float(v), 4) for v in 2 * p * r / np.maximum(p + r, 1e-12)]


def resolve_revision(name):
    """Pin the exact Hub commit so the run is traceable. Local paths return None."""
    if Path(name).exists():
        return None
    try:
        from huggingface_hub import HfApi
        return HfApi().model_info(name).sha
    except Exception as e:
        log(f"WARN could not resolve revision for {name}: {e}")
        return None


def dir_hash(d):
    h = hashlib.sha256()
    for p in sorted(Path(d).rglob("*")):
        if p.is_file():
            h.update(p.name.encode())
            h.update(p.read_bytes())
    return h.hexdigest()


def get_normalizer(kind):
    if kind == "basic":
        return basic_norm
    if kind == "csebuet":
        # BanglaBERT's model card requires this normalizer. Fail loudly rather than fall back.
        try:
            from normalizer import normalize as bn_normalize
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "git+https://github.com/csebuetnlp/normalizer"], check=True)
            from normalizer import normalize as bn_normalize
        return lambda t: bn_normalize(basic_norm(t))
    raise ValueError(kind)


class TextClassifier(nn.Module):
    """Encoder -> masked mean pooling -> dropout -> Linear(3). Text in, logits out. No other inputs."""

    def __init__(self, backbone, dropout, msd):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.msd = msd
        self.head = nn.Linear(backbone.config.hidden_size, 3)

    def forward(self, input_ids, attention_mask):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        pooled = (h * m).sum(1) / m.sum(1).clamp_min(1.0)
        if self.training and self.msd > 1:
            return torch.stack([self.head(self.drop(pooled)) for _ in range(self.msd)]).mean(0)
        return self.head(self.drop(pooled))


def param_groups(model, cfg):
    """Layer-wise LR decay: top encoder layer gets cfg['lr'], each layer below x cfg['llrd']."""
    n_layers = model.backbone.config.num_hidden_layers
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight", "LayerNorm.bias")
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("head."):
            lr = cfg["head_lr"]
        else:
            m = re.search(r"encoder\.layer\.(\d+)\.", name)
            depth = int(m.group(1)) + 1 if m else 0          # embeddings/other = depth 0
            if "pooler" in name:
                depth = n_layers
            lr = cfg["lr"] * cfg["llrd"] ** (n_layers - depth)
        wd = 0.0 if any(nd in name for nd in no_decay) else cfg["wd"]
        groups.setdefault((lr, wd), []).append(p)
    return [dict(params=ps, lr=lr, weight_decay=wd) for (lr, wd), ps in groups.items()]


def make_batch(tok_ids, rows, pad_id):
    L = max(len(tok_ids[i]) for i in rows)
    inp = torch.full((len(rows), L), pad_id, dtype=torch.long)
    att = torch.zeros((len(rows), L), dtype=torch.long)
    for r, i in enumerate(rows):
        s = tok_ids[i]
        inp[r, :len(s)] = torch.tensor(s, dtype=torch.long)
        att[r, :len(s)] = 1
    return inp.to(DEVICE), att.to(DEVICE)


@torch.no_grad()
def predict(model, tok_ids, rows, pad_id, bs=128):
    model.eval()
    rows = np.asarray(rows)
    order = np.argsort([len(tok_ids[i]) for i in rows], kind="stable")
    out = np.zeros((len(rows), 3), dtype=np.float32)
    for s in range(0, len(rows), bs):
        pos = order[s:s + bs]
        inp, att = make_batch(tok_ids, rows[pos], pad_id)
        with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
            logits = model(inp, att)
        out[pos] = torch.softmax(logits.float(), -1).cpu().numpy()
    return out


def atomic_savez(path, **arrays):
    """Write to a temp file in the same folder, then rename, so a disconnect never leaves half a file."""
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "wb") as fh:
        np.savez(fh, **arrays)
    os.replace(tmp, path)


def atomic_json(path, obj):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    os.replace(tmp, path)


def rows_for(cfg, k):
    """Global train-row indices for fold k (train/val) and test rows covered by this run."""
    tr, va = np.flatnonzero(folds != k), np.flatnonzero(folds == k)
    te = np.arange(N_TE)
    if cfg["subset"] == "bangla":
        tr, va, te = tr[BANGLA_TR[tr]], va[BANGLA_TR[va]], te[BANGLA_TE]
    elif cfg["subset"] is not None:
        raise ValueError(cfg["subset"])
    return tr, va, te


def config_hash(name, cfg):
    payload = json.dumps(dict(name=name, cfg=cfg, data=DATA_HASHES, seed=SEED), sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()


def valid_probs(P, n):
    return (P.shape == (n, 3) and np.isfinite(P).all() and (P >= 0).all() and (P <= 1).all()
            and float(np.abs(P.sum(1) - 1).max()) < 1e-3)


def fold_file_ok(fp, name, cfg, k):
    """Skip-if-valid: a saved fold counts only if every check passes; otherwise it is retrained."""
    if not fp.exists():
        return False
    try:
        z = np.load(fp, allow_pickle=False)
        tr, va, te = rows_for(cfg, k)
        ok = (str(z["config_hash"]) == config_hash(name, cfg)
              and np.array_equal(z["va"], va) and np.array_equal(z["te_rows"], te)
              and valid_probs(z["vp"], len(va)) and valid_probs(z["tp"], len(te)))
        if not ok:
            log(f"[{name}] fold {k}: saved file failed validation -> retraining")
        return bool(ok)
    except Exception as e:
        log(f"[{name}] fold {k}: unreadable saved file ({e}) -> retraining")
        return False


def run_fold(name, cfg, k, tok_ids_tr, tok_ids_te, pad_id, revision, smoke=False):
    tr, va, te = rows_for(cfg, k)
    rng = np.random.RandomState(SEED + k)
    if smoke:
        tr, va, te = rng.choice(tr, 1200, replace=False), va[:300], te[:300]
    torch.manual_seed(SEED + k)
    if CUDA:
        torch.cuda.manual_seed_all(SEED + k)
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False

    backbone = AutoModel.from_pretrained(cfg["model"], revision=revision)
    model = TextClassifier(backbone, cfg["dropout"], cfg["msd"]).to(DEVICE)
    opt = torch.optim.AdamW(param_groups(model, cfg))
    epochs = 1 if smoke else cfg["epochs"]
    steps_per_epoch = math.ceil(len(tr) / cfg["bs"] / cfg["grad_accum"])
    sched = get_linear_schedule_with_warmup(opt, int(cfg["warmup"] * steps_per_epoch * epochs),
                                            steps_per_epoch * epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=CUDA)
    counts = np.bincount(y[tr], minlength=3)
    cw = torch.tensor(len(tr) / (3 * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)

    hist, vps, tps, t0 = [], [], [], time.time()
    for ep in range(epochs):
        model.train()
        perm = rng.permutation(len(tr))
        run_loss, nb = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for bi, s in enumerate(range(0, len(tr), cfg["bs"])):
            pos = perm[s:s + cfg["bs"]]
            inp, att = make_batch(tok_ids_tr, tr[pos], pad_id)
            yy = torch.tensor(y[tr[pos]], dtype=torch.long, device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
                logits = model(inp, att)
            loss = F.cross_entropy(logits.float(), yy, weight=cw,
                                   label_smoothing=cfg["label_smoothing"]) / cfg["grad_accum"]
            if not torch.isfinite(loss):
                raise RuntimeError(f"non-finite loss at fold {k} epoch {ep + 1} batch {bi}")
            scaler.scale(loss).backward()
            if (bi + 1) % cfg["grad_accum"] == 0 or s + cfg["bs"] >= len(tr):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
            run_loss += float(loss) * cfg["grad_accum"]
            nb += 1
        vp = predict(model, tok_ids_tr, va, pad_id)
        f1 = macro_f1(y[va], vp.argmax(1))
        hist.append(dict(epoch=ep + 1, loss=run_loss / max(nb, 1), val_macro_f1=f1,
                         val_per_class=per_class_f1(y[va], vp.argmax(1)), sec=round(time.time() - t0)))
        log(f"  [{name}] fold {k} epoch {ep + 1}/{epochs} loss={run_loss / max(nb, 1):.4f} "
            f"val macroF1={f1:.4f} ({time.time() - t0:.0f}s)  [reporting only, not used for selection]")
        if ep >= epochs - cfg["avg_last"]:
            vps.append(vp)
            tps.append(predict(model, tok_ids_te, te, pad_id))
    vp_avg, tp_avg = np.mean(vps, 0).astype(np.float32), np.mean(tps, 0).astype(np.float32)
    vp_last, tp_last = vps[-1], tps[-1]
    state = None
    if cfg["save_state"] and not smoke:
        state = {k_: v.half().cpu() for k_, v in model.state_dict().items()}
    del model, backbone, opt
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()
    return dict(tr=tr, va=va, te=te, vp=vp_avg, tp=tp_avg, vp_last=vp_last, tp_last=tp_last,
                hist=hist, state=state)


def run_model(name, smoke=False):
    cfg = RUNS[name]
    final_fp = RUN_DIR / f"{name}_final.npz"
    chash = config_hash(name, cfg)
    revision = resolve_revision(cfg["model"])
    tok = AutoTokenizer.from_pretrained(cfg["model"], revision=revision)
    pad_id = tok.pad_token_id
    assert pad_id is not None, "tokenizer has no pad token"
    tok_dir = RUN_DIR / f"_tok_{name}"
    tok.save_pretrained(tok_dir)
    meta = dict(name=name, cfg=cfg, config_hash=chash, model_revision=revision,
                tokenizer_sha256=dir_hash(tok_dir), data=DATA_HASHES, seed=SEED,
                versions=dict(torch=torch.__version__, transformers=transformers.__version__,
                              numpy=np.__version__, pandas=pd.__version__,
                              python=platform.python_version()),
                gpu=torch.cuda.get_device_name(0) if CUDA else None, uses_id=False)

    norm = get_normalizer(cfg["norm"])
    need_tr = np.flatnonzero(BANGLA_TR) if cfg["subset"] == "bangla" else np.arange(N_TR)
    need_te = np.flatnonzero(BANGLA_TE) if cfg["subset"] == "bangla" else np.arange(N_TE)
    tok_ids_tr, tok_ids_te = [None] * N_TR, [None] * N_TE
    enc_tr = tok([norm(train["text"].iat[i]) for i in need_tr], truncation=True,
                 max_length=cfg["max_len"])["input_ids"]
    enc_te = tok([norm(test["text"].iat[i]) for i in need_te], truncation=True,
                 max_length=cfg["max_len"])["input_ids"]
    for i, e in zip(need_tr, enc_tr):
        tok_ids_tr[i] = e
    for i, e in zip(need_te, enc_te):
        tok_ids_te[i] = e
    trunc = float(np.mean([len(e) >= cfg["max_len"] for e in enc_tr]))
    meta["truncated_share_train"] = round(trunc, 4)
    log(f"[{name}] model={cfg['model']} rev={revision} rows train={len(need_tr)} test={len(need_te)} "
        f"truncated@{cfg['max_len']}={trunc:.3%} config_hash={chash[:12]}")

    if smoke:
        out = run_fold(name, cfg, 0, tok_ids_tr, tok_ids_te, pad_id, revision, smoke=True)
        assert valid_probs(out["vp"], len(out["va"])) and valid_probs(out["tp"], len(out["te"]))
        log(f"[{name}] SMOKE TEST OK (nothing saved)")
        return None

    for k in range(N_SPLITS):
        fp = RUN_DIR / f"{name}_fold{k}.npz"
        if fold_file_ok(fp, name, cfg, k):
            log(f"[{name}] fold {k} valid on disk -> skipped")
            continue
        out = run_fold(name, cfg, k, tok_ids_tr, tok_ids_te, pad_id, revision)
        atomic_savez(fp, va=out["va"], vp=out["vp"], tp=out["tp"], te_rows=out["te"],
                     vp_last=out["vp_last"], tp_last=out["tp_last"],
                     hist=np.array(json.dumps(out["hist"])), config=np.array(json.dumps(cfg)),
                     config_hash=np.array(chash))
        if out["state"] is not None:
            tmp = RUN_DIR / f"{name}_fold{k}_state.pt.tmp"
            torch.save(out["state"], tmp)
            os.replace(tmp, RUN_DIR / f"{name}_fold{k}_state.pt")
        assert fold_file_ok(fp, name, cfg, k), "fold file failed validation right after writing"
        log(f"[{name}] fold {k} SAVED  val macroF1={macro_f1(y[out['va']], out['vp'].argmax(1)):.4f}")

    # stitch folds -> final arrays. Rows outside a subset run's coverage stay NaN.
    oof = np.full((N_TR, 3), np.nan, dtype=np.float32)
    test_sum, fold_scores, hists = None, [], []
    cov_tr = np.zeros(N_TR, bool)
    for k in range(N_SPLITS):
        z = np.load(RUN_DIR / f"{name}_fold{k}.npz", allow_pickle=False)
        assert np.array_equal(np.sort(z["va"]), rows_for(cfg, k)[1])
        oof[z["va"]] = z["vp"]
        cov_tr[z["va"]] = True
        test_sum = z["tp"].astype(np.float64) if test_sum is None else test_sum + z["tp"]
        fold_scores.append(macro_f1(y[z["va"]], z["vp"].argmax(1)))
        hists.append(json.loads(str(z["hist"])))
        te_rows = z["te_rows"]
    test_p = np.full((N_TE, 3), np.nan, dtype=np.float32)
    test_p[te_rows] = (test_sum / N_SPLITS).astype(np.float32)
    cov_te = np.zeros(N_TE, bool)
    cov_te[te_rows] = True
    assert valid_probs(oof[cov_tr], cov_tr.sum()) and valid_probs(test_p[cov_te], cov_te.sum())
    assert np.isnan(oof[~cov_tr]).all() and np.isnan(test_p[~cov_te]).all()

    score = macro_f1(y[cov_tr], oof[cov_tr].argmax(1))
    meta.update(oof_macro_f1_covered=score, per_class=per_class_f1(y[cov_tr], oof[cov_tr].argmax(1)),
                fold_scores=fold_scores, fold_std=float(np.std(fold_scores)),
                coverage_train=int(cov_tr.sum()), coverage_test=int(cov_te.sum()), epoch_history=hists,
                test_pred_share=np.bincount(test_p[cov_te].argmax(1), minlength=3).tolist())
    atomic_savez(final_fp, oof=oof, test=test_p, fold_ids=folds, coverage_train=cov_tr,
                 coverage_test=cov_te, meta=np.array(json.dumps(meta, default=str)))
    atomic_json(RUN_DIR / f"{name}_meta.json", meta)
    log(f"[{name}] DONE  OOF macroF1 (covered rows {cov_tr.sum()}) = {score:.6f}  "
        f"per-class {meta['per_class']}  folds {[round(s, 4) for s in fold_scores]}")
    return meta

In [ ]:
# ============ CELL 5: SMOKE TEST (~3 min, saves nothing). Run once before the long runs. ============
for _name in RUN_NAMES:
    run_model(_name, smoke=True)

In [ ]:
# ============ CELL 6: FULL RUNS (long; safe to re-run after any disconnect) ============
RESULTS = {}
for _name in RUN_NAMES:
    RESULTS[_name] = run_model(_name)

In [ ]:
# ============ CELL 7: REPORT (reads only from Drive, so it works after a restart) ============
for fp in sorted(RUN_DIR.glob("*_meta.json")):
    m = json.loads(fp.read_text(encoding="utf-8"))
    print(f"{m['name']:16s} OOF {m['oof_macro_f1_covered']:.6f}  per-class {m['per_class']}  "
          f"folds {[round(s, 4) for s in m['fold_scores']]}  std {m['fold_std']:.4f}  "
          f"covered {m['coverage_train']}/47817  uses_id={m['uses_id']}")
print("\nNo submission file was created. Run the next cell and take the ZIP to the local blend step.")

In [ ]:
# ============ CELL 8: EXPORT ZIP (self-contained: defines its own paths, needs no earlier cell) ============
import os, json, time, hashlib, zipfile
from pathlib import Path

_ROOT = Path(os.environ.get("UAP_WORK_ROOT", "/content/drive/MyDrive/uap_text"))
if str(_ROOT).startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
_RUNS, _EXPORT = _ROOT / "runs", _ROOT / "exports"
_EXPORT.mkdir(parents=True, exist_ok=True)
_files = sorted([p for p in _RUNS.glob("*.npz")] + [p for p in _RUNS.glob("*_meta.json")]
                + [p for p in [_ROOT / "progress.log", _ROOT / "data" / "fold_ids_colab.npy"] if p.exists()])
_files = [p for p in _files if not p.name.endswith(".tmp")]
assert _files, f"nothing to export in {_RUNS}"


def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


_manifest = {p.name: dict(sha256=_sha(p), size=p.stat().st_size) for p in _files}
_zip = _EXPORT / f"uap_text_export_{time.strftime('%Y%m%d_%H%M%S')}.zip"
with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in _files:
        zf.write(p, arcname=("runs/" if p.parent == _RUNS else "") + p.name)
    zf.writestr("export_manifest.json", json.dumps(_manifest, indent=2))
with zipfile.ZipFile(_zip) as zf:
    assert zf.testzip() is None
print(f"wrote {_zip} ({_zip.stat().st_size / 1e6:.1f} MB, {len(_files)} files + manifest)")
try:
    from google.colab import files
    files.download(str(_zip))
except Exception:
    print("download it from Drive:", _zip)